# AK Video 1 — Google Colab GPU training
This runs the existing random-initialized AK Video 1 prototype. It is not a polished 15–30 second generator yet. Add only videos you own or are licensed to use, with accurate captions.

**Required Drive layout:** `MyDrive/AK-Video-1/data/captions.csv` and `MyDrive/AK-Video-1/data/videos/<filename>`.

CSV columns: `filename,caption`. Keep at least one separate validation clip; the current loader makes a small holdout split automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/sidharthkumar8580-oss/Ankit-Kryptex-.git /content/Ankit-Kryptex
%cd /content/Ankit-Kryptex
!pip -q install -r video_model/requirements.txt


In [ ]:
import os, torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then rerun.')
DATA='/content/drive/MyDrive/AK-Video-1/data'
assert os.path.isfile(DATA+'/captions.csv'), 'Upload captions.csv and videos/ folder to MyDrive/AK-Video-1/data first.'
assert os.path.isdir(DATA+'/videos'), 'Missing videos/ folder.'


In [ ]:
# Small smoke test first. Saves checkpoints directly to Google Drive.
!python -m video_model.train --data "$DATA" --steps 100 --batch-size 1 --save-every 50 --save /content/drive/MyDrive/AK-Video-1/ak-video-1-smoke.pt


## Longer training
After the smoke test finishes without errors, run the next cell. Free Colab sessions can disconnect; resume from the latest checkpoint using `--resume`. The model is a tiny 16-frame, 64×64 research baseline. A small dataset will overfit and will not produce realistic general-purpose video.

In [ ]:
!python -m video_model.train --data "$DATA" --steps 10000 --batch-size 1 --save-every 250 --resume --save /content/drive/MyDrive/AK-Video-1/ak-video-1.pt


In [ ]:
# Generate a silent test clip only after training has saved a checkpoint.
!python -m video_model.generate --checkpoint /content/drive/MyDrive/AK-Video-1/ak-video-1.pt --prompt "A red bicycle moving along a quiet road" --output /content/ak-video-1-test.mp4
from IPython.display import Video, display
display(Video('/content/ak-video-1-test.mp4', embed=True))
